# Salary Prediction

**Tabular Regression Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `salary`

## 2 · Project Overview

We predict **employee salary** based on years of experience, education level, department, age, certifications, company size, and performance rating.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Encode categorical features for tree-based and linear models.
3. Build a baseline Linear Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with RMSE, MAE, R², and residual analysis.

## 4 · Problem Statement

Given an employee's experience, education, department, age, certifications, company size, and performance rating, predict their annual salary.

## 5 · Why This Project Matters

- **Compensation benchmarking** is critical for HR and recruiting.
- Fair pay analysis helps detect salary inequities.
- Salary prediction supports budgeting and hiring decisions.
- Classic regression with ordinal and categorical features.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 6,000 |
| **Features** | years_experience, education, department, age, certifications, company_size, performance_rating |
| **Target** | salary (continuous, USD) |
| **Range** | ~$20K – $250K |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "salary"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: salary
Save dir: E:\Github\Machine-Learning-Projects\Regression\Salary Prediction


## 11 · Dataset Download or Loading

Synthetic dataset: 6,000 employees with career and demographic features.

In [4]:
np.random.seed(SEED)
n = 6000
years_experience = np.round(np.random.exponential(7, n).clip(0, 35), 1)
education = np.random.choice(["High School", "Bachelor", "Master", "PhD"], n,
                              p=[0.15, 0.45, 0.3, 0.1])
edu_bonus = {"High School": 0, "Bachelor": 10000, "Master": 22000, "PhD": 35000}
edu_val = np.array([edu_bonus[e] for e in education], dtype=float)

department = np.random.choice(["Engineering", "Sales", "Marketing", "HR", "Finance", "Operations"], n,
                               p=[0.25, 0.2, 0.15, 0.1, 0.15, 0.15])
dept_bonus = {"Engineering": 15000, "Finance": 12000, "Marketing": 5000,
              "Sales": 8000, "HR": 3000, "Operations": 6000}
dept_val = np.array([dept_bonus[d] for d in department], dtype=float)

age = np.round(22 + years_experience + np.random.normal(3, 2, n), 0).clip(18, 65).astype(int)
certifications = np.random.poisson(1.5, n).clip(0, 8)
company_size = np.random.choice(["Small", "Medium", "Large"], n, p=[0.3, 0.4, 0.3])
size_bonus = np.where(company_size == "Large", 8000, np.where(company_size == "Medium", 3000, 0))
performance_rating = np.random.uniform(1, 5, n).round(1)

salary = (30000 + 3000 * years_experience + edu_val + dept_val + size_bonus
          + 1500 * certifications + 2000 * performance_rating
          + np.random.normal(0, 5000, n))
salary = np.round(salary, -2).clip(20000, 250000)

df = pd.DataFrame({"years_experience": years_experience, "education": education,
                    "department": department, "age": age, "certifications": certifications,
                    "company_size": company_size, "performance_rating": performance_rating,
                    "salary": salary})
print(f"Dataset shape: {df.shape}")
print(f"Target stats:\n{df['salary'].describe()}")
df.head()

Dataset shape: (6000, 8)
Target stats:
count      6000.000000
mean      86012.716667
std       23946.375121
min       30800.000000
25%       69400.000000
50%       81700.000000
75%       97900.000000
max      199900.000000
Name: salary, dtype: float64


,years_experience,education,department,age,certifications,company_size,performance_rating,salary
0,3.3,Master,HR,26,2,Small,1.9,67700.0
1,21.1,Bachelor,Engineering,50,0,Medium,1.1,118300.0
2,9.2,Master,Engineering,38,1,Medium,1.3,107700.0
3,6.4,Master,Finance,31,3,Small,3.8,91700.0
4,1.2,Bachelor,Marketing,25,0,Medium,1.7,50500.0


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")
print(f"\nTarget stats:\n{df[TARGET].describe()}")

DATA VALIDATION

Shape: (6000, 8)

Missing values:
years_experience      0
education             0
department            0
age                   0
certifications        0
company_size          0
performance_rating    0
salary                0
dtype: int64

Duplicate rows: 0

Dtypes:
years_experience      float64
education              object
department             object
age                     int64
certifications          int32
company_size           object
performance_rating    float64
salary                float64
dtype: object

Target 'salary' confirmed.

Target stats:
count      6000.000000
mean      86012.716667
std       23946.375121
min       30800.000000
25%       69400.000000
50%       81700.000000
75%       97900.000000
max      199900.000000
Name: salary, dtype: float64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0][0].scatter(df["years_experience"], df[TARGET], alpha=0.2, s=10)
axes[0][0].set_xlabel("Years Experience"); axes[0][0].set_ylabel("Salary"); axes[0][0].set_title("Experience vs Salary")

df.boxplot(column=TARGET, by="education", ax=axes[0][1])
axes[0][1].set_title("Salary by Education")

df.boxplot(column=TARGET, by="department", ax=axes[0][2])
axes[0][2].set_title("Salary by Department")
axes[0][2].tick_params(axis="x", rotation=45)

df.boxplot(column=TARGET, by="company_size", ax=axes[1][0])
axes[1][0].set_title("Salary by Company Size")

axes[1][1].scatter(df["performance_rating"], df[TARGET], alpha=0.2, s=10)
axes[1][1].set_xlabel("Performance Rating"); axes[1][1].set_ylabel("Salary"); axes[1][1].set_title("Performance vs Salary")

corr = df.select_dtypes(include="number").corr()
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", center=0, ax=axes[1][2], square=True)
axes[1][2].set_title("Correlation")

plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `salary`.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[TARGET], bins=30, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}")
axes[0].set_xlabel("Salary ($)")

df.groupby("education")[TARGET].median().reindex(["High School","Bachelor","Master","PhD"]).plot(
    kind="bar", ax=axes[1], color="steelblue", edgecolor="black")
axes[1].set_title("Median Salary by Education")
axes[1].set_ylabel("Salary ($)")

plt.tight_layout()
plt.show()
print(f"Range: ${df[TARGET].min():,.0f} to ${df[TARGET].max():,.0f}")
print(f"Mean: ${df[TARGET].mean():,.0f}, Median: ${df[TARGET].median():,.0f}")

Range: $30,800 to $199,900
Mean: $86,013, Median: $81,700


## 15 · Train/Validation/Test Split Strategy

80/20 train/test split.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target — Train mean: {y_train.mean():,.2f}, Test mean: {y_test.mean():,.2f}")

Train: (4800, 7), Test: (1200, 7)
Target — Train mean: 85,994.27, Test mean: 86,086.50


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `education`, `department`, `company_size` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Outliers**: High salaries (executives) — keep them.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["exp_per_cert"] = X_train["years_experience"] / (X_train["certifications"] + 1)
X_test["exp_per_cert"] = X_test["years_experience"] / (X_test["certifications"] + 1)

X_train["age_exp_diff"] = X_train["age"] - X_train["years_experience"]
X_test["age_exp_diff"] = X_test["age"] - X_test["years_experience"]

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (9): ['years_experience', 'education', 'department', 'age', 'certifications', 'company_size', 'performance_rating', 'exp_per_cert', 'age_exp_diff']


## 18 · Baseline Model

Linear Regression as our baseline regressor.

In [10]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

BASELINE: Linear Regression
RMSE : 8,438.52
MAE  : 6,606.93
R2   : 0.8782


## 19 · LazyPredict Benchmark

Fit dozens of regressors in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Regressors:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Regressors:
                               Adjusted R-Squared  R-Squared         RMSE  Time Taken
Model                                                                                
CatBoostRegressor                        0.954665   0.955005  5128.682008    3.224740
GradientBoostingRegressor                0.953995   0.954341  5166.401742    0.535728
HistGradientBoostingRegressor            0.951161   0.951528  5323.166955    2.610332
LGBMRegressor                            0.950956   0.951324  5334.312956    0.066419
ExtraTreesRegressor                      0.943824   0.944246  5709.006545    1.061950
XGBRegressor                             0.942147   0.942581  5793.613202    0.169575
RandomForestRegressor                    0.940176   0.940625  5891.468200    1.415951
BaggingRegressor                         0.935604   0.936088  6112.454431    0.150877
AdaBoostRegressor                        0.904107   0.904827  7458.997213    0.266881
KNeighborsRegressor  

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred_flaml):,.2f}")
print(f"R2  : {r2_score(y_test, y_pred_flaml):.4f}")

FLAML best model: catboost
RMSE: 5,039.18
R2  : 0.9566


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Linear Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost RMSE: 5,001.27  (1.8s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[143]	valid_0's l2: 2.82383e+07
LightGBM RMSE: 5,313.97  (14.8s)


XGBoost failed: XGBModel.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by RMSE):
                      RMSE      MAE      R2
CatBoost           5001.27  3999.88  0.9572
FLAML              5039.18  4023.80  0.9566
LightGBM           5313.97  4228.72  0.9517
Linear Regression  8438.52  6606.93  0.8782

Top 3 models: ['CatBoost', 'FLAML', 'LightGBM']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)

    axes[i].scatter(y_test, yp, alpha=0.6, s=40, edgecolors="black")
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_predicted_vs_actual.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")


Detailed Metrics — Top 3:

  CatBoost:
    RMSE : 5,001.27
    MAE  : 3,999.88
    R2   : 0.9572

  FLAML:
    RMSE : 5,039.18
    MAE  : 4,023.80
    R2   : 0.9566

  LightGBM:
    RMSE : 5,313.97
    MAE  : 4,228.72
    R2   : 0.9517


## 24 · Error Analysis

Examine residuals from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=20, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.6, s=40, edgecolors="black")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

sorted_idx = np.argsort(best_pred)
axes[2].scatter(best_pred[sorted_idx], np.abs(residuals[sorted_idx]), alpha=0.6, s=40, edgecolors="black")
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")
print(f"  Median: {np.median(residuals):,.2f}")

Residual stats (CatBoost):
  Mean: 14.35, Std: 5,001.25
  Min: -16,940.71, Max: 16,348.13
  Median: -131.73


## 25 · Interpretation and Business Insight

**Key findings:**
- **Years of experience** is the strongest salary predictor.
- **Education level** provides clear step-function increases (HS → Bachelor → Master → PhD).
- **Department** matters — Engineering and Finance pay highest.
- **Company size** adds a premium (Large > Medium > Small).
- **Performance rating** has a moderate positive effect.

**Business takeaway:** Experience + Education + Department explain most salary variance. Company size and performance add incremental signal.

## 26 · Limitations

1. Synthetic data with additive wage model.
2. No job title or seniority level.
3. Missing negotiation and market demand factors.
4. No geographic cost-of-living adjustment.
5. Gender/race not included (important for equity analysis).

## 27 · How to Improve This Project

1. Add job title/seniority ladder.
2. Include geographic location and COL index.
3. Use real salary survey data (Glassdoor, BLS).
4. Add industry sector.
5. Model salary growth trajectories over time.

## 28 · Production Considerations

- Deploy for automated compensation benchmarking.
- Output salary bands (prediction intervals).
- Monitor for pay equity across demographics.
- Update annually with market data.
- Integrate with HRIS systems.

## 29 · Common Mistakes

1. Ignoring multicollinearity between age and experience.
2. Treating education as continuous instead of ordinal.
3. Not log-transforming right-skewed salaries.
4. Including prohibited factors (race, gender) in pricing.
5. Overfitting to one company's data.

## 30 · Mini Challenge / Exercises

1. Log-transform `salary` — does Linear Regression improve?
2. Remove `age` (correlated with experience) — any change?
3. Build separate models by department.
4. Plot salary growth curves by education level.
5. Calculate the ROI of a Master's degree vs. 3 years experience.

## 31 · Final Summary / Key Takeaways

1. **Experience** is the #1 salary predictor.
2. **Education** provides step-function increases.
3. **Department** reflects market demand differentials.
4. **Company size** adds a consistent premium.
5. **Performance** has moderate impact (less than structural factors).
6. **Salary equity** analysis requires demographic data not in this dataset.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Regression\Salary Prediction\metrics.json
{
  "CatBoost": {
    "rmse": 5001.27,
    "mae": 3999.88,
    "r2": 0.9572
  },
  "LightGBM": {
    "rmse": 5313.97,
    "mae": 4228.72,
    "r2": 0.9517
  },
  "Linear Regression": {
    "rmse": 8438.52,
    "mae": 6606.93,
    "r2": 0.8782
  },
  "FLAML": {
    "rmse": 5039.18,
    "mae": 4023.8,
    "r2": 0.9566
  }
}
